# Q_2

**1. The model shows strong performance on validation data but fails on new matches recorded in different clubs with different camera heights and angles. Hit detection accuracy drops significantly.
Explain the most likely root causes of this failure and describe how you would systematically investigate and isolate the problem.**

El problema más probable es que el modelo esté haciendo overfitting a las condiciones presentes en los datos de entrenamiento y validación en cuanto a posición y altura de la cámara, ángulo, iluminación, tipo de pista, etc. \
Además, el problema puede estar en la estimación de la pose o en la clasificación del golpe. Un error en los keypoints puede propagarse al clasificador y hacer que parezca que el problema viene de ese cuando en realidad se ha originado en la primera etapa. 

Lo investigaría de la siguiente forma:
1. Descomponer el rendimiento por diferentes factores, es decir, analizaría las métricas separando por club, cámara, altura, ángulo, jugador y tipo de golpe. Si el rendimiento del modelo se ve afectado por el ángulo o la altura de la cámara, pero no al cambiar de club o jugador, el problema sería geométrico. 
2. Separar pose estimation de hit classification, midiendo la calidad de los keypoints en los nuevos partidos y comprobar si el error de hit detection está correlacionado con errores en muñecas, codos o hombors. De esta forma se puede localizar en que fase de produce el error. 
3. Realizar experimentos controlados con transformaciones geométricas de forma que simulen cambios de cámara, como rotaciones, escalado, recortes y transformaciones de perspectiva. Si el rendimiento baja, sería una evidencia de que el modelo es sensible al cambio geométrico. 
4. Validación cruzada por dominio. Si existen partidos de diferentes clubs o cámaras, utilizaría splits del tipo leave-one-out, de forma que entrenaría con determinados grupos y evaluaría en otros. De esta forma se puede medir la capacidad de generalización del modelo. 
5. Comprobar si hay posibles fugas de información en el split, ya que si hay frames del mismo partido, jugador o cámara tanto en train como en validation, las métricas obtenidas no son fiables. 

**2. After reviewing some samples, you discover that different annotators labeled keypoints slightly differently (especially wrists and elbows), and hit timestamps are sometimes shifted by a few frames.
Analyze how inconsistent annotations affect both pose estimation and hit classification, how you would detect label-quality problems at scale and how you would decide whether to relabel, filter, or reweight parts of the dataset. Justify your decision criteria.**

Las anotaciones inconsistentes pueden afectar tanto a la estimación de la pose como a la detección de golpes, y además hacer que las métricas de validación y test sean poco fiables. 

En pose estimation distinguiría dos tipos de error:
* Ruido aleatorio, donde el keypoint está ligeramente desplazado respecto a la posición correcta.
* Sesgo sistemático, donde por ejemplo un anotador coloca sistemáticamente la muñeca unos pixeles por encima de su posición real. En este caso el modelo puede aprender un patrón asociado al anotador.

En la parte temporal, un desplazamiento de unos frames en los timestamps introduce una etiqueta ruidosa del instante del impacto. Si el modelo aprende a partir de ventanas temporales, esto puede hacer que secuencias prácticamente idénticas tengan etiquetas diferentes, además de afectar a métricas derivadas del movimiento realizado. 

Además, si estos errores están también en validation o test, el problema no solamente es que el modelo no aprenda, si no que las métricas no representan la calidad real del modelo. 


Para detectar problemas de calidad mediría la consistencia entre anotadores mediante una muestra anotada por varias personas. Para los keypoints utilizaría métricas como distancia entre anotadores normalizada por el tamaño del jugador, y para los eventos mediría la distribución de diferencias temporales entre anotadores. 

En cuanto a la decisión de relabel, filtrar o reponderar, no aplicaría una única estrategia a todo el dataset. 
* Error sistemático y corregible: corregir automáticamente si se ha caracterizado el sesgo. Por ejemplo, si existe un offset temporal sistemático de un anotador, se puede corregir.
* Errores en validation o test: reanotación, ya que estas muestras determinan las métricas con las que se evalúa el sistema. 
* Keypoints críticos mal anotados: si afectan a muñecas o codos y tienen impacto en clases que se confunden entre sí, los revisaría. 
* Ruido residual que no merece el coste de reanotación: se pueden utilizar pesos por muestra o por anotador para reducir su influencia durante el entrenamiento. 

Antes de reanotar todo el dataset haría un experimento piloto sobre una muestra representativa de forma que pueda comparar el rendimiento con las etiquetas actuales y las corregidas para estimar cuánto mejora el modelo. Si la mejora merece el coste de anotación, lo aplicaría; en caso contrario, utilizaría corrección o reponderación. 

Es decir, la decisión la basaria según la magnitud del error, el impacto sobre el modelo y el coste de corregirlo. 

**3. Smashes are rare in the dataset but very important for the product. The model often misses them while performing well on common hits.
Without changing the model architecture, explain which dataset, training, and evaluation strategy changes you would consider. Here, the idea is to discuss tradeoffs and risks of each approach and how you would verify that improvements are real and not overfitting artifacts.**

El problema puede ser causado por pocos ejemplos de smash, poca diversidad de esos ejemplos, etiquetas ambiguas o dificultad para distinguirlo de otros golpes. 

Cambios en el dataset:
1. Añadir más smashes reales: en caso de ser posible añadiría más ejemplos con diferentes jugadores, cámaras y situaciones. No añadiría únicamente muestras del mismo jugador o partido, ya que no aumentaría la diversidad de las muestras. 
2. Data augmentation para smashes: aplicaría transformaciones moderadas de tipo geométricas, escalado, cambios de perspectiva o ruido en los keypoints. El riesgo es que una transformación demasiado agresiva puede cambiar el movimiento y convertirlo en una muestra poco realista o en otra clase de golpe. Por esto sería necesario validar cada augmentation. 
3. Datos sintéticos: sería útil para aumentar la diversidad a bajo coste, pero con el riesgo de introducir gap respecto a los datos reales. Además, no incluiría datos sintéticos en validation y test. 

Cambios en el entrenamiento:
* Class weighted loss: aumentar el peso de los errores sobre la clase smash. Es barato computacionalmente, pero un peso excesivo puede provocar falsos positivos. 
* Focal loss: útil si los ejemplos difíciles son los que no está aprendiendo, pero requiere ajustar hiperparámetros. 
* Two stage training: se entrenaría el modelo con el dataset completo y luego haría fine-tuning con una distribución más balanceada. Esto puede ayudar a que la clase smash tenga suficiente influencia, aunque aumenta el coste de entrenamiento. 
* Features derivadas de keypoints: por ejemplo se podría introducir altura y posición relativa o velocidad, en caso de que pueda ayudar a diferenciar un smash de otro tipo de golpe, aunque cambiaria el input del modelo. 

Cambios en evaluación:\
No utilizaría accuracy global, ya que no representa correctamente todas las clases al estar desbalanceadas, si no que miraría:
* precision, recall y F1-score por clase
* matriz de confusión
* precision-recall curve

El objetivo es conseguir un buen trade-off entre recall y precision. 

\
Para comprobar que hay una mejora en el modelo, mantendría el conjunto de test completamente independiente. \
Aplicaría cambios de forma incremental para detectar que está causando la mejora, y además los repetiría con varias semillas. \
Además, es necesario comprobar que la mejora en la detección de smash no degrada otras clases. 

**4. Two training runs with the same configuration produce noticeably different results. One model is stable but less accurate; another is more accurate but produces jittery keypoints and inconsistent hit timing.
Provide a reasoning to decide which model is actually better for production. What additional tests, metrics, and scenario-based evaluations would you design to support the decision?**

La diferencia entre las dos ejecuciones puede deberse a la aleatoriedad del entrenamiento como: inicialización de pesos, orden de los batches, augmentations aleatorias, dropout, etc.

Además, hay que distinguir entre accuracy espacial y estabilidad temporal, ya que un modelo puede tener mejor accuracy frame a frame pero producir trayectorias de keypoints poco coherentes. 

El jitter de los keypoints es también relevante, ya que el clasificador utiliza el movimiento del cuerpo como señal. Si la trayectoria de la muñeca o el codo tiene ruido, este puede propagarse a la detección del golpe y producir timestamps inconsistentes, duplicar eventos o falsos positivos. \
El timestamp también es importante, ya que es necesario detectar el momento en el que ha ocurrido el evento. 

Por tanto, a priori favorecería el modelo más estable, pero no basaría la decisión únicamente en la accuracy. 

\
Pruebas adicionales antes de tomar la decisión:
1. Calidad de los keypoints\
Medir el error de los keypoints y el jitter, como la variación de la trayectoria frame a frame respecto al movimiento real esperado. \
Además probaría a aplicar perturbaciones del input como ruido, cambios de iluminación o transformaciones geométricas. Un modelo robusto debería soportar estos pequeños cambios. 
2. Calidad de hit detection\
Medir precision, recall y F1-score por tipo de golpe, además del error temporal del impacto o rendimiento en diferentes cámaras. 
3. Variabilidad entre seeds\
Entrenar varias veces cada configuración y comparar la distribución de los resultados, no solamente un run. Un modelo un poco peor pero estable es preferible. 
4. Evaluación con y sin postprocesado temporal\
Aplicaría el mismo filtro temporal a ambos modelos. Si el modelo más preciso mantiene su ventaja y el filtrado elimina el jitter sin degradar el timing, podría ser la mejor opción. \
Pero, hay que tener en cuenta que el suavizado introduce latencia y puede desplazar el momento del impacto, por lo que hay que medir el efecto sobre el timestamp. 

\
Criterio de decisión
* Si la diferencia de accuracy desaparece dentro de la variabilidad entre seeds, elegiría el modelo más estable. 
* Si el modelo más preciso solamente mejora en clases frecuentes, elegiría el modelo más estable.
* Si mejora claramente las clases críticas y mantiene buen timing, elegiría el modelo más preciso. 
* Si el jitter puede corregirse mediante un postprocesado que no introduce una latencia significativa, el modelo preciso puede ser la mejor opción. 
* Si el modelo preciso genera timestamps inestables, elegiría el modelo estable. 

**5. Which model would you propose for this application? Why?**

Para esta aplicación propondría un pipeline en dos etapas: pose estimation con tracking y clasificador temporal sobre secuencias de keypoints. \
La idea es identificar el movimiento corporal asociado a un golpe a partir de la zona de la imagen donde se produce. 

\
Pipeline
1. Detección y tracking del jugador\
Utilizaría un detector pequeño como YOLO y un tracker como ByteTrack para mantener la identidad del jugador entre frames. \
El tracking permite construir una secuencia temporal consistente del jugador que queremos analizar. 
2. Pose estimation\
Utilizaría un modelo ligero, como una variante pequeña de RTMPose, y aplicaría cuantización si se puede. De esta forma se obtiene una buena relación entre precisión y coste computacional. 
3. Filtrado temporal\
Aplicaría un postprocesado temporal sobre los keypoints para reducir el jitter. Además, utilizaría las confianzas de los keypoints para identificar posiciones poco fiables. 
4. Normalización de la pose\
Transformaría las coordenadas para que no dependan de la posición del jugador en la imagen. Por ejemplo, se podría normalizar respecto al centro del cuerpo. \
Además, usaría features derivadas de los keypoints, como posiciones relativas o altura normalizada. De esta forma, se puede reducir la dependencia de la cámara para que el clasificador aprenda correctamente el movimiento. 
5. Clasificador temporal\
Sobre una ventana de tiempo (1-2 segundos) utilizaría un modelo temporal como una TCN 1D o una GRU pequeña para clasificar el movimiento.

\
Este enfoque tiene un coste menor que usar modelo de video end-to-end. Además, usar la representación basada en el esqueleto elimina información como el fondo, color, iluminación o ropa, de forma que hacemos que el sistema sea más robusto. \
En el caso del clasificador, no estamos observando únicamente un frame, sino que la evolución de la pose es lo que permite distinguir los golpes entre ellos. \
El pipeline permite separar ambos problemas, de forma que si hay algún error es más fácil de identificar. 